# 3B — Random Forest Regressor

**Project:** Predictive Modeling of US Used Vehicle Prices  
**Course:** ENGR422 — Applied Machine Learning  
**Author:** *(assign one team member)*  

---

This notebook covers the **Random Forest Regressor** from **Work Package 3 — Model Implementation**.

**Deliverable:** D3.2 — Tuned Ensemble Models (Random Forest portion)

## 3B.1 — Imports & Load Data

Import necessary libraries. Load the **raw** (unprocessed) train/test splits from Notebook 02,
plus the **fitted preprocessor pipeline** (`preprocessor_tree.pkl`).

We pre-transform the data once up front for fast baseline experiments and CV.
The full `Pipeline(preprocessor + model)` is assembled only at the end (§3B.9)
for the final `.pkl` export — that way the saved model can accept raw data directly.

In [ ]:
# 3B.1 — Imports & Load Data
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time

# Import custom transformers so joblib.load() can resolve the classes
from preprocessing import (
    OdometerBinnedYearImputer,
    YearBinnedOdometerImputer,
    OdometerBinnedConditionImputer,
    ModelBinnedManufacturerImputer,
    CascadingCategoricalImputer,
    MeanTargetEncoder,
)

RANDOM_STATE = 42

# Load raw (unprocessed) train/test splits
X_train = pd.read_csv("../data/X_train.csv")
X_test = pd.read_csv("../data/X_test.csv")
y_train = pd.read_csv("../data/y_train.csv").values.ravel()
y_test = pd.read_csv("../data/y_test.csv").values.ravel()

# Load the fitted preprocessing pipeline from Notebook 02
preprocessor_tree = joblib.load("../models/preprocessor_tree.pkl")

# Pre-transform once for fast baseline experiments & CV
X_train_processed = preprocessor_tree.transform(X_train)
X_test_processed = preprocessor_tree.transform(X_test)

print(f"X_train raw shape:       {X_train.shape}")
print(f"X_train processed shape: {X_train_processed.shape}")
print(f"X_test processed shape:  {X_test_processed.shape}")
print(f"y_train shape:           {y_train.shape}")
print(f"y_test shape:            {y_test.shape}")


## 3B.2 — Model Setup

Initialize the `RandomForestRegressor` with sensible baseline defaults.
We cap `max_depth=20` to keep trees from growing enormous on 305K rows,
and set `n_jobs=-1` here (but `n_jobs=1` on CV later to avoid CPU oversubscription).

In [ ]:
# 3B.2 — Model Setup
rf_baseline = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

print(f"Baseline model: {rf_baseline}")


## 3B.3 — Baseline Training (Default Hyperparameters)

Train the baseline Random Forest on pre-transformed data. Then run 5-fold CV.

**Performance note:** `n_jobs=-1` is on the model (parallel trees), so CV runs with `n_jobs=1`
to avoid oversubscription. Pre-transforming the data also eliminates repeated preprocessing per fold.

In [ ]:
# 3B.3 — Baseline Training (Default Hyperparameters)
print("Training baseline Random Forest on pre-transformed data...")
start_time = time.time()
rf_baseline.fit(X_train_processed, y_train)
train_time = time.time() - start_time
print(f"Training completed in {train_time:.2f} seconds")

# n_jobs=1 on CV because the RF already uses all cores internally
print("\nPerforming 5-fold cross-validation...")
start_time = time.time()

cv_mae = cross_val_score(rf_baseline, X_train_processed, y_train, cv=5,
                         scoring="neg_mean_absolute_error", n_jobs=1)
print(f"CV MAE:  {-cv_mae.mean():,.2f} +/- {cv_mae.std():,.2f}")

cv_rmse = cross_val_score(rf_baseline, X_train_processed, y_train, cv=5,
                          scoring="neg_root_mean_squared_error", n_jobs=1)
print(f"CV RMSE: {-cv_rmse.mean():,.2f} +/- {cv_rmse.std():,.2f}")

cv_time = time.time() - start_time
print(f"\nCV completed in {cv_time:.2f} seconds")


## 3B.4 — Hyperparameter Tuning

Use `RandomizedSearchCV` to tune key hyperparameters:
- `n_estimators`: 100–400
- `max_depth`: 15–35
- `min_samples_split`: 2, 3, 5, 8 (includes sklearn default of 2)
- `min_samples_leaf`: 1, 2, 3, 5 (includes sklearn default of 1)
- `max_features`: sqrt, log2, 0.3, 0.5, 0.7, None (None = all features, same as baseline default)

20 iterations x 3-fold CV = 60 fits. Estimated ~20-30 min on a local machine.

In [ ]:
# 3B.4 — Hyperparameter Tuning (RandomizedSearchCV)
from scipy.stats import randint

param_distributions = {
    'n_estimators':      randint(100, 400),
    'max_depth':         randint(15, 35),
    'min_samples_split': [2, 3, 5, 8],       # 2 = sklearn default (no extra constraint)
    'min_samples_leaf':  [1, 2, 3, 5],        # 1 = sklearn default (no extra constraint)
    'max_features':      ['sqrt', 'log2', 0.3, 0.5, 0.7, None],  # None = all features
}

random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_distributions,
    n_iter=20,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=1,
    random_state=RANDOM_STATE,
    verbose=2,
    return_train_score=True,
)

print("Starting RandomizedSearchCV (20 iterations x 3 folds = 60 fits)...")
print("Estimated time: ~20-30 minutes on a local machine.\n")
start_time = time.time()
random_search.fit(X_train_processed, y_train)
search_time = time.time() - start_time

print(f"\nSearch completed in {search_time/60:.1f} minutes")
print(f"\nBest parameters:")
for param, val in random_search.best_params_.items():
    print(f"  {param}: {val}")
print(f"\nBest CV MAE: {-random_search.best_score_:,.2f}")


## 3B.5 — Train Best Model

Retrain with the best hyperparameters found during tuning. Compare against the baseline
using 5-fold CV on equal footing, and automatically select whichever model performs better.

In [ ]:
# 3B.5 — Train Best Model
# Compare tuned vs baseline, pick the better one
best_params = random_search.best_params_

rf_tuned = RandomForestRegressor(
    **best_params,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

print("Training tuned model with best hyperparameters...")
start_time = time.time()
rf_tuned.fit(X_train_processed, y_train)
train_time = time.time() - start_time
print(f"Training completed in {train_time:.2f} seconds")

# 5-fold CV for both models on equal footing
print("\nPerforming 5-fold CV with tuned params...")
cv_mae_tuned = cross_val_score(rf_tuned, X_train_processed, y_train, cv=5,
                                scoring="neg_mean_absolute_error", n_jobs=1)

tuned_score = -cv_mae_tuned.mean()
baseline_score = -cv_mae.mean()

print(f"CV MAE (tuned):    {tuned_score:,.2f} +/- {cv_mae_tuned.std():,.2f}")
print(f"CV MAE (baseline): {baseline_score:,.2f} +/- {cv_mae.std():,.2f}")

# Pick the winner
if tuned_score < baseline_score:
    rf_best = rf_tuned
    print(f"\nTuned model wins by ${baseline_score - tuned_score:,.2f} MAE. Using tuned model.")
else:
    rf_best = rf_baseline
    print(f"\nBaseline model is equal or better. Using baseline model.")
    print("(This can happen when default params are already near-optimal for this dataset.)")


## 3B.6 — Test Set Evaluation

Predict on the held-out test set and compute all four project metrics:
- **MAE** (Mean Absolute Error)
- **RMSE** (Root Mean Squared Error)
- **R\u00b2** (R-squared)
- **MAPE** (Mean Absolute Percentage Error)

Plot predicted vs. actual prices scatter plot and the residuals distribution.

In [ ]:
# 3B.6 — Test Set Evaluation
y_pred = rf_best.predict(X_test_processed)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)

print("=== Test Set Metrics ===")
print(f"  MAE:  ${mae:,.2f}")
print(f"  RMSE: ${rmse:,.2f}")
print(f"  R2:   {r2:.4f}")
print(f"  MAPE: {mape:.2%}")

# --- Predicted vs Actual scatter ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

idx = np.random.RandomState(RANDOM_STATE).choice(len(y_test), size=min(5000, len(y_test)), replace=False)

axes[0].scatter(y_test[idx], y_pred[idx], alpha=0.3, s=10, color="steelblue")
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             "r--", linewidth=1.5, label="Perfect prediction")
axes[0].set_xlabel("Actual Price ($)")
axes[0].set_ylabel("Predicted Price ($)")
axes[0].set_title("Predicted vs Actual")
axes[0].legend()

# --- Residuals distribution ---
residuals = y_test - y_pred
axes[1].hist(residuals, bins=80, color="steelblue", edgecolor="white", alpha=0.8)
axes[1].axvline(0, color='r', linestyle='--', linewidth=1.5)
axes[1].set_xlabel("Residual (Actual - Predicted)")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Residuals Distribution (mean={residuals.mean():,.0f}, std={residuals.std():,.0f})")

plt.tight_layout()
plt.show()


## 3B.7 — Feature Importance

Extract and visualize feature importances from the Random Forest.
Feature names are taken directly from the preprocessed DataFrame columns
(preserved by `set_output(transform="pandas")` in the preprocessing pipeline).

In [ ]:
# 3B.7 — Feature Importance
# Feature names come directly from the preprocessed DataFrame columns.

feature_names = X_train_processed.columns.tolist()
importances = rf_best.feature_importances_

# Sort by importance
sorted_idx = np.argsort(importances)
top_n = min(20, len(feature_names))
top_idx = sorted_idx[-top_n:]

# Clean up column names for display (strip transformer prefix)
display_names = [n.split('__', 1)[-1] if '__' in n else n for n in feature_names]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(top_n), importances[top_idx], color='steelblue', alpha=0.8)
ax.set_yticks(range(top_n))
ax.set_yticklabels([display_names[i] for i in top_idx])
ax.set_xlabel("Feature Importance (MDI)")
ax.set_title(f"Top {top_n} Feature Importances - Random Forest")
plt.tight_layout()
plt.show()

# Print top 10 as a table
print("\nTop 10 features:")
for rank, i in enumerate(sorted_idx[::-1][:10], 1):
    print(f"  {rank}. {display_names[i]:<30s}  {importances[i]:.4f}")


## 3B.8 — Overfitting Analysis

Compare training set performance vs. test set performance to check for overfitting.

In [ ]:
# 3B.8 — Overfitting Analysis

# Training set metrics
y_train_pred = rf_best.predict(X_train_processed)
train_mae  = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_r2   = r2_score(y_train, y_train_pred)
train_mape = mean_absolute_percentage_error(y_train, y_train_pred)

# Comparison table
print('=== Train vs Test Performance ===')
print(f'{"Metric":<8s}  {"Train":>12s}  {"Test":>12s}  {"Gap":>10s}')
print('-' * 46)
print(f'{"MAE":<8s}  ${train_mae:>10,.2f}  ${mae:>10,.2f}  ${mae - train_mae:>8,.2f}')
print(f'{"RMSE":<8s}  ${train_rmse:>10,.2f}  ${rmse:>10,.2f}  ${rmse - train_rmse:>8,.2f}')
print(f'{"R2":<8s}  {train_r2:>11.4f}   {r2:>11.4f}   {r2 - train_r2:>9.4f}')
print(f'{"MAPE":<8s}  {train_mape:>11.2%}   {mape:>11.2%}   {mape - train_mape:>9.2%}')

gap_pct = ((mae - train_mae) / train_mae) * 100
print(f'\nMAE gap: {gap_pct:.1f}% higher on test set')
if gap_pct > 50:
    print('Warning: Significant overfitting detected.')
elif gap_pct > 20:
    print('Moderate overfitting — model generalizes reasonably but has room for improvement.')
else:
    print('Low overfitting — model generalizes well.')


## 3B.9 — Save Model

Compose the fitted preprocessor and the best Random Forest into a single `Pipeline`,
then serialize it to `models/random_forest.pkl`. This way, Notebook 04 can call
`.predict()` on raw (unprocessed) data directly.

In [ ]:
# 3B.9 — Save Model
full_pipeline = Pipeline([
    ("preprocess", preprocessor_tree),
    ("model", rf_best),
])

# Sanity check: predict on raw X_test (not pre-transformed)
y_pred_check = full_pipeline.predict(X_test)
check_mae = mean_absolute_error(y_test, y_pred_check)
print(f"Full pipeline MAE on raw test data: ${check_mae:,.2f}")
print(f"Direct model MAE (pre-transformed): ${mae:,.2f}")
assert np.isclose(check_mae, mae, rtol=1e-5), 'Pipeline and direct predictions should match!'
print("Pipeline predictions match direct predictions\n")

joblib.dump(full_pipeline, "../models/random_forest.pkl")
print("Saved: ../models/random_forest.pkl")
